# Z boson mass from CMS open dimuon data

## 1. What the data is
The data is a public CMS file of 100000 collision events from 2011. Each event has two muons. For each muon the file gives the energy and the momentum components, the transverse momentum (pt), the pseudorapidity (eta), the charge, and a type: global (G) or tracker (T).

An invariant mass is the mass of a parent particle worked out from its two decay products. For two muons I add the four-vectors and compute `M = sqrt(E^2 - px^2 - py^2 - pz^2)` of the sum. If both muons come from one decaying particle, M equals that particle's mass. A Z boson decays to two muons often enough that a pile of events appears near 91 GeV. Other events have two muons that are unrelated, and those form a smooth background.

The file also has an M column. I recompute M from the four-vectors and use my own value everywhere. Everything below reads only `data/`, `results/` and `config.py`.

In [1]:
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display
import config
import dimuon

plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
BLUE, ORANGE, GREY, RED = "#2a6fbb", "#d9822b", "#777777", "#b03a2e"
raw = pd.read_csv(config.DATA_FILE)
df = dimuon.add_recomputed_mass(raw)
meta = json.load(open("data/metadata.json"))
print("rows:", len(df), "| file size (bytes):", meta["file_size_bytes"], "| sha256:", meta["sha256"][:16], "...")
share = float(np.mean(np.abs(df["mass"] - df["M"]) > config.M_DIFF_TOLERANCE))
print(f"share of rows where the file's M differs from my recomputed mass by more than {config.M_DIFF_TOLERANCE} GeV: {share:.5f}")


rows: 100000 | file size (bytes): 13935840 | sha256: 58caa45c580f8bcf ...
share of rows where the file's M differs from my recomputed mass by more than 0.01 GeV: 0.00198


## 2. The full dimuon mass spectrum
Log-log axes, 0.3 to 300 GeV. Only the J/psi and the Z are labelled, because those are the two peaks this project fits.

In [2]:
edges = np.logspace(np.log10(0.3), np.log10(300.0), 301)
counts, _ = np.histogram(df["mass"], bins=edges)
widths = np.diff(edges)
centres = np.sqrt(edges[1:] * edges[:-1])
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.step(edges[:-1], counts / widths, where="post", color=BLUE, lw=1.1)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("dimuon invariant mass (GeV)"); ax.set_ylabel("events per GeV")
ax.set_xlim(0.3, 300)
jp = counts[(centres > 2.9) & (centres < 3.3)].argmax()
ymax_j = (counts / widths)[(centres > 2.9) & (centres < 3.3)].max()
ymax_z = (counts / widths)[(centres > 80) & (centres < 100)].max()
ax.annotate("J/psi", xy=(config.PDG_MJPSI, ymax_j), xytext=(config.PDG_MJPSI * 0.55, ymax_j * 4), arrowprops=dict(arrowstyle="-", color=GREY), color=GREY)
ax.annotate("Z", xy=(config.PDG_MZ, ymax_z), xytext=(config.PDG_MZ * 1.15, ymax_z * 4), arrowprops=dict(arrowstyle="-", color=GREY), color=GREY)
ax.set_ylim(top=ymax_j * 100)
ax.set_title("All 100000 events in the file (recomputed mass)")
fig.tight_layout(); fig.savefig("Images/dimuon_spectrum.png", dpi=150); plt.close(fig)
print("events with recomputed mass below 0.3 or above 300 GeV:", int(((df['mass'] < 0.3) | (df['mass'] >= 300)).sum()))
print("saved Images/dimuon_spectrum.png")


events with recomputed mass below 0.3 or above 300 GeV: 24
saved Images/dimuon_spectrum.png


![spectrum](Images/dimuon_spectrum.png)

## 3. Cutflow

In [3]:
cf = pd.read_csv("results/cutflow.csv")
display(cf.pivot(index="step", columns="selection", values="events"))
print(cf.to_string(index=False))


selection,Jpsi,Z,scan
step,,,
0,100000.0,100000.0,100000.0
1,93474.0,93474.0,93474.0
2,93474.0,93474.0,93474.0
3,93474.0,93474.0,93474.0
4,7270.0,5752.0,31121.0
5,NaN,5064.0,NaN


selection  step                    cut  events
        Z     0     all events in file  100000
        Z     1      both muons global   93474
        Z     2        opposite charge   93474
        Z     3   |eta| < 2.4 for both   93474
     Jpsi     0     all events in file  100000
     Jpsi     1      both muons global   93474
     Jpsi     2        opposite charge   93474
     Jpsi     3   |eta| < 2.4 for both   93474
     scan     0     all events in file  100000
     scan     1      both muons global   93474
     scan     2        opposite charge   93474
     scan     3   |eta| < 2.4 for both   93474
        Z     4 pt > 20.0 GeV for both    5752
        Z     5  mass in [70.0, 110.0)    5064
     Jpsi     4     mass in [2.9, 3.3)    7270
     scan     4   mass in [15.0, 60.0)   31121


## 4. The nominal Z fit
Unbinned extended maximum likelihood. Voigt signal with the width fixed to the reference value, exponential background. The curve is drawn on a histogram of 0.5 GeV bins only for display. The fit itself uses every event.

In [4]:
zf = json.load(open("results/z_fit_nominal.json"))
sel = dimuon.z_selection(df)
mz = df.loc[sel, "mass"].to_numpy()
a, b = config.Z_RANGE
model = dimuon.build_model("z_fixed_width", a, b, len(mz))
p = zf["values"]
bedges = np.arange(a, b + 1e-9, config.Z_BIN_WIDTH)
cnt, _ = np.histogram(mz, bins=bedges)
es, eb = dimuon.bin_expected(model, p, bedges)
mu = es + eb
bc = 0.5 * (bedges[1:] + bedges[:-1])
grid = np.linspace(a, b, 800)
fs, fb = dimuon.model_density_components(model, grid, p)
fig, (ax, axp) = plt.subplots(2, 1, figsize=(7.5, 6), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
ax.errorbar(bc, cnt, yerr=np.sqrt(np.maximum(cnt, 1)), fmt="o", ms=3, color="black", label="data")
ax.plot(grid, (fs + fb) * config.Z_BIN_WIDTH, color=RED, lw=1.8, label="total fit")
ax.plot(grid, fb * config.Z_BIN_WIDTH, color=BLUE, ls="--", lw=1.5, label="background")
ax.set_ylabel("events per 0.5 GeV"); ax.legend(frameon=False)
pull = np.where(mu > 0, (cnt - mu) / np.sqrt(np.maximum(mu, 1e-9)), 0.0)
axp.bar(bc, pull, width=config.Z_BIN_WIDTH, color=GREY)
axp.axhline(0, color="black", lw=0.8); axp.set_ylabel("pull"); axp.set_xlabel("dimuon invariant mass (GeV)")
fig.tight_layout(); fig.savefig("Images/z_fit.png", dpi=150); plt.close(fig)
print("saved Images/z_fit.png")
print(f"events in the fit: {zf['n_events']}")
print(f"mass = {zf['mass']:.4f} +/- {zf['stat_error']:.4f} (stat, hesse) GeV; minos {zf['minos']['lower']:+.4f} / {zf['minos']['upper']:+.4f}")
print(f"sigma = {zf['values']['sigma']:.4f} +/- {zf['hesse_errors']['sigma']:.4f} GeV, lam = {zf['values']['lam']:.4f}")
print(f"Ns = {zf['values']['Ns']:.1f}, Nb = {zf['values']['Nb']:.1f}")
print(f"valid: {zf['valid']}, parameter near a limit: {zf['at_limit_params']}, accurate covariance: {zf['accurate_covariance']}")
g = zf["gof"]
print(f"goodness of fit: chi2 = {g['chi2']:.2f}, ndof = {g['ndof']}, p = {g['p_value']:.4f} ({g['n_bins_used']} of {g['n_bins']} bins used)")
print(f"difference from reference {zf['reference_mass']}: {zf['difference_GeV']:+.4f} GeV = {zf['difference_in_stat_se']:+.2f} stat. standard errors = {zf['difference_in_total_se']:+.2f} total standard errors")


saved Images/z_fit.png
events in the fit: 5064
mass = 90.7905 +/- 0.0398 (stat, hesse) GeV; minos -0.0398 / +0.0398
sigma = 1.3100 +/- 0.0542 GeV, lam = -0.0496
Ns = 4616.6, Nb = 447.4
valid: True, parameter near a limit: [], accurate covariance: True
goodness of fit: chi2 = 102.05, ndof = 72, p = 0.0114 (77 of 80 bins used)
difference from reference 91.1879: -0.3974 GeV = -9.98 stat. standard errors = -5.86 total standard errors


![z fit](Images/z_fit.png)

**Reading the fit.** The fit is valid, with no parameter near a limit and an accurate covariance. The fitted mass is 90.7905 GeV with a hesse error of 0.0398 GeV, and minos gives the same interval. That is 0.3974 GeV below the reference value, which is 9.98 statistical standard errors and 5.86 total standard errors. The fitted mass does not agree with the reference value within its errors.

The pull panel shows a run of positive pulls between about 80 and 86 GeV, on the low-mass side of the peak, and the goodness-of-fit p-value is 0.0114. An excess on the low side is what final-state radiation would produce, and the fit model has no radiation. A likely reason for part of the shift is therefore the missing radiation. An uncalibrated muon momentum scale is another likely reason. Neither is proven here.

## 5. Systematic variants
Each variant changes one thing from the nominal fit. The detector-region row is shown but is not added to the systematic total, because it is a different sample and its difference includes statistical fluctuation.

In [5]:
sy = pd.read_csv("results/z_systematics.csv")
print(sy[["group", "description", "events", "mass", "error", "shift_from_nominal", "valid", "at_limit_params"]].to_string(index=False))
print()
for k, v in zf["group_systematics"].items():
    print(f"group {k}: {v:.4f} GeV")
print(f"indicative systematic estimate (quadrature sum of the five groups): {zf['indicative_systematic']:.4f} GeV")
print(f"statistical error: {zf['stat_error']:.4f} GeV, total error: {zf['total_error']:.4f} GeV")
fig, ax = plt.subplots(figsize=(8, 4.8))
y = np.arange(len(sy))[::-1]
cols = [GREY if g.startswith("6") else BLUE for g in sy["group"]]
ax.axvspan(zf["mass"] - zf["stat_error"], zf["mass"] + zf["stat_error"], color=BLUE, alpha=0.12, label="nominal, stat. error")
ax.axvline(zf["mass"], color=BLUE, lw=1)
ax.axvline(config.PDG_MZ, color=RED, lw=1.5, label="reference value")
ax.errorbar(sy["mass"], y, xerr=sy["error"], fmt="o", ms=4, color="black", ecolor=GREY, capsize=2)
ax.set_yticks(y); ax.set_yticklabels([f"{g.split(' ',1)[0]}: {d}" for g, d in zip(sy["group"], sy["description"])], fontsize=8)
ax.set_xlabel("fitted Z mass (GeV)"); ax.legend(frameon=False, loc="upper left", fontsize=8)
fig.tight_layout(); fig.savefig("Images/z_systematics.png", dpi=150); plt.close(fig)
print("saved Images/z_systematics.png")


                           group               description  events      mass    error  shift_from_nominal  valid at_limit_params
                         nominal               nominal fit    5064 90.790538 0.039796            0.000000   True             NaN
              1 background shape         linear background    5064 90.801777 0.039924            0.011238   True               c
              1 background shape   second-order background    5064 90.802554 0.039924            0.012015   True              c1
                     2 fit range       range 75 to 105 GeV    4904 90.810315 0.039812            0.019776   True             NaN
                     2 fit range       range 66 to 116 GeV    5140 90.775478 0.039841           -0.015060   True             NaN
             3 muon pt threshold               pt > 25 GeV    4582 90.814806 0.041044            0.024268   True             NaN
             3 muon pt threshold               pt > 30 GeV    3878 90.840410 0.044030            

saved Images/z_systematics.png


![systematics](Images/z_systematics.png)

**Reading the systematics.** The largest group is the muon pt threshold at 0.0499 GeV, and it comes from the 30 GeV variant, which keeps only 3878 events. Part of that shift is probably sample fluctuation and not only a modelling effect. In the linear and second-order background variants one background shape parameter sits close to its limit, and in the 30 GeV variant the background yield does. The Minuit flag still reports those fits as valid. The second-order background is negative near the upper edge of the range at the fitted values (lowest value -0.105), so that variant is left out of the systematic estimate. The table still shows it.

The indicative systematic estimate is 0.0548 GeV, larger than the statistical error of 0.0398 GeV. Every variant moves the mass by at most 0.0499 GeV, which is much smaller than the 0.3974 GeV difference from the reference. None of the modelling choices I varied explains that difference. The detector-region row differs from nominal by 0.0015 GeV, but it is a different sample with 2162 events and is not part of the total.

## 6. Toy study of the nominal fit
200 toys from the fitted nominal model. The pull is (fitted mass minus true mass) divided by the hesse error.

In [6]:
tt = pd.read_csv("results/z_toys.csv")
ts = json.load(open("results/z_toys_summary.json"))
print(f"toys: {ts['n_toys']}, invalid fits: {ts['n_invalid']}")
print(f"pull mean = {ts['pull_mean']:.3f} +/- {ts['pull_mean_err']:.3f}, pull standard deviation = {ts['pull_std']:.3f}")
fig, ax = plt.subplots(figsize=(6, 3.8))
good = tt[tt.valid]
ax.hist(good.pull, bins=np.linspace(-4, 4, 25), density=True, color=BLUE, alpha=0.7)
xx = np.linspace(-4, 4, 200)
ax.plot(xx, stats.norm.pdf(xx), color="black", lw=1, label="standard normal")
ax.set_xlabel("pull of the fitted mass"); ax.set_ylabel("density"); ax.legend(frameon=False)
ax.set_title(f"mean {ts['pull_mean']:.2f}, width {ts['pull_std']:.2f}")
fig.tight_layout(); fig.savefig("Images/z_toy_pulls.png", dpi=150); plt.close(fig)


toys: 200, invalid fits: 0
pull mean = -0.010 +/- 0.067, pull standard deviation = 0.950


![toy pulls](Images/z_toy_pulls.png)

**Reading the toys.** The pull has a mean of -0.010 with an uncertainty of 0.067 and a standard deviation of 0.950, and none of the 200 toy fits was invalid. The fit recovers the true mass without a visible bias, and the hesse error is close to the real spread. This tests the fit machinery against its own model. It says nothing about how well the model describes the real data.

## 7. J/psi fit and the two-mass scale check
Crystal Ball signal (tail on the low-mass side, n fixed) plus exponential background. The Gaussian signal is the variant. The relative scale is (fitted mass minus reference) divided by reference. This checks the momentum scale at two masses. It is not a precision measurement.

In [7]:
jf = json.load(open("results/jpsi_fit.json"))
sj = dimuon.jpsi_selection(df)
mj = df.loc[sj, "mass"].to_numpy()
ja, jb = config.JPSI_RANGE
jm = dimuon.build_model("jpsi_cb", ja, jb, len(mj))
pj = jf["crystal_ball"]["values"]
jedges = np.arange(ja, jb + 1e-9, config.JPSI_BIN_WIDTH)
jc, _ = np.histogram(mj, bins=jedges)
jcent = 0.5 * (jedges[1:] + jedges[:-1])
jes, jeb = dimuon.bin_expected(jm, pj, jedges)
jmu = jes + jeb
jg = np.linspace(ja, jb, 800)
js, jb_ = dimuon.model_density_components(jm, jg, pj)
fig, (ax, axp) = plt.subplots(2, 1, figsize=(7.5, 6), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
ax.errorbar(jcent, jc, yerr=np.sqrt(np.maximum(jc, 1)), fmt="o", ms=2.5, color="black", label="data")
ax.plot(jg, (js + jb_) * config.JPSI_BIN_WIDTH, color=RED, lw=1.6, label="total fit (Crystal Ball)")
ax.plot(jg, jb_ * config.JPSI_BIN_WIDTH, color=BLUE, ls="--", lw=1.3, label="background")
ax.axvline(config.PDG_MJPSI, color=GREY, lw=1, ls=":", label="reference mass")
ax.set_ylabel("events per 0.005 GeV"); ax.legend(frameon=False)
axp.bar(jcent, (jc - jmu) / np.sqrt(np.maximum(jmu, 1e-9)), width=config.JPSI_BIN_WIDTH, color=GREY)
axp.axhline(0, color="black", lw=0.8); axp.set_ylabel("pull"); axp.set_xlabel("dimuon invariant mass (GeV)")
fig.tight_layout(); fig.savefig("Images/jpsi_fit.png", dpi=150); plt.close(fig)
print("saved Images/jpsi_fit.png")
print(f"J/psi events: {len(mj)}")
for key in ["crystal_ball", "gaussian"]:
    r = jf[key]
    print(f"{key}: mass = {r['mass']:.5f} +/- {r['mass_err']:.5f} GeV, sigma = {r['values']['sigma']:.5f}, valid = {r['valid']}, near limit: {r['at_limit_params']}, chi2/ndof = {r['gof']['chi2']:.1f}/{r['gof']['ndof']}, p = {r['gof']['p_value']:.4g}")
print("crystal ball beta =", round(jf["crystal_ball"]["values"]["beta"], 3))
sc = pd.read_csv("results/scale_check.csv")
sc["relative_scale_percent"] = 100 * sc["relative_scale"]
sc["relative_scale_stat_error_percent"] = 100 * sc["relative_scale_stat_error"]
print(sc[["resonance", "fitted_mass", "fit_error", "reference", "relative_scale_percent", "relative_scale_stat_error_percent", "valid"]].to_string(index=False))


saved Images/jpsi_fit.png
J/psi events: 7270
crystal_ball: mass = 3.09346 +/- 0.00047 GeV, sigma = 0.02988, valid = True, near limit: [], chi2/ndof = 124.8/74, p = 0.0002066
gaussian: mass = 3.09297 +/- 0.00045 GeV, sigma = 0.03045, valid = True, near limit: [], chi2/ndof = 143.3/75, p = 3.497e-06
crystal ball beta = 1.725
               resonance  fitted_mass  fit_error  reference  relative_scale_percent  relative_scale_stat_error_percent  valid
    J/psi (Crystal Ball)     3.093459   0.000475     3.0969               -0.111113                           0.015325   True
J/psi (Gaussian variant)     3.092966   0.000450     3.0969               -0.127025                           0.014534   True
             Z (nominal)    90.790538   0.039796    91.1879               -0.435761                           0.043642   True


![jpsi](Images/jpsi_fit.png)

**Reading the scale check.** The J/psi fit with the Crystal Ball gives 3.09346 GeV, which is 0.111 percent below the reference, and the Gaussian variant gives 0.127 percent below. The Z fit is 0.436 percent below its reference. All three offsets are negative, but the Z offset is larger in relative terms, so a single multiplicative momentum scale does not describe both masses in this analysis.

The J/psi fit quality is poor, with p-values of 0.0002 (Crystal Ball) and 3.5e-06 (Gaussian). The reference masses are also defined differently from a fitted peak position. Both facts make the J/psi number a rough check. It is not a precision measurement of the momentum scale.

## 8. The bump scan and the look-elsewhere effect
The region 15 to 60 GeV has no known resonance I am fitting. A smooth four-parameter background is fitted to 0.5 GeV bins. A Gaussian of relative width equal to the Z resolution is moved across 87 test masses. The most significant local fluctuation is then compared with the same scan run on 500 background-only pseudo-experiments.

In [8]:
bf = json.load(open("results/scan_background_fit.json"))
sl = pd.read_csv("results/scan_local.csv")
st = pd.read_csv("results/scan_toys.csv")
ss = json.load(open("results/scan_summary.json"))
ssel = dimuon.scan_selection(df)
cnt_s = dimuon.scan_counts(df.loc[ssel, "mass"].to_numpy())
ed = dimuon.scan_edges(); cc = 0.5 * (ed[1:] + ed[:-1])
curve = dimuon.background_curve(bf["parameters"], cc)
fig, axs = plt.subplots(1, 3, figsize=(14, 4))
axs[0].errorbar(cc, cnt_s, yerr=np.sqrt(cnt_s), fmt="o", ms=2.5, color="black")
axs[0].plot(cc, curve, color=RED, lw=1.5); axs[0].set_yscale("log")
axs[0].set_xlabel("dimuon invariant mass (GeV)"); axs[0].set_ylabel("events per 0.5 GeV"); axs[0].set_title("scan region and background fit")
axs[1].plot(sl["test_mass"], sl["z_local"], color=BLUE, lw=1.3)
k = int(sl["z_local"].idxmax())
axs[1].plot(sl["test_mass"][k], sl["z_local"][k], "o", color=RED)
axs[1].set_xlabel("test mass (GeV)"); axs[1].set_ylabel("local significance"); axs[1].set_title("local significance")
axs[2].hist(st["max_z_local"], bins=np.linspace(0, max(st["max_z_local"].max(), ss["observed_max_z_local"]) + 0.3, 25), color=BLUE, alpha=0.75)
axs[2].axvline(ss["observed_max_z_local"], color=RED, lw=2, label="observed")
axs[2].set_xlabel("largest local significance in a pseudo-experiment"); axs[2].set_ylabel("pseudo-experiments"); axs[2].legend(frameon=False)
axs[2].set_title("500 background-only toys")
fig.tight_layout(); fig.savefig("Images/bump_scan.png", dpi=150); plt.close(fig)
print("saved Images/bump_scan.png")
print(f"events in the scan region: {ss['n_events_scan']}, background parameters: {[round(v, 4) for v in bf['parameters']]}")
print(f"background-only goodness of fit: chi2 = {ss['background_gof_chi2']:.2f}, ndof = {ss['background_gof_ndof']}, p = {ss['background_gof_p_value']:.4f}, model poor (p < {ss['gof_alpha']}): {ss['background_model_poor']}")
print(f"relative resolution r = sigma_Z / mass_Z = {ss['relative_resolution_r']:.5f}")
print(f"largest local significance: {ss['observed_max_z_local']:.3f} at test mass {ss['observed_mass']} GeV (fitted S = {ss['observed_S']:.1f})")
print(f"local p-value: {ss['p_local']:.4f}")
print(f"toys with a larger or equal maximum: {ss['n_toys_ge_observed']} of {ss['n_toys']}")
print(f"global p-value: {ss['p_global']:.4f}")
print(f"trials factor (p_global / p_local): {ss['trials_factor']:.2f}")
print(f"toy maxima: median {ss['toy_max_z_median']:.2f}, 95th percentile {ss['toy_max_z_95pct']:.2f}")
print(f"invalid scan fits: observed {ss['n_invalid_scan_fits_observed']}, toys {ss['n_invalid_scan_fits_toys']}")


saved Images/bump_scan.png
events in the scan region: 31121, background parameters: [4.9121, -2.2816, 0.3573, 0.0005]
background-only goodness of fit: chi2 = 66.97, ndof = 86, p = 0.9361, model poor (p < 0.01): False
relative resolution r = sigma_Z / mass_Z = 0.01443
largest local significance: 1.961 at test mass 26.0 GeV (fitted S = 77.2)
local p-value: 0.0250
toys with a larger or equal maximum: 394 of 500
global p-value: 0.7884
trials factor (p_global / p_local): 31.58
toy maxima: median 2.25, 95th percentile 3.12
invalid scan fits: observed 0, toys 0


![scan](Images/bump_scan.png)

**Reading the scan.** The largest local significance is 1.961 at a test mass of 26.0 GeV, with a local p-value of 0.0250. Among the 500 background-only toys, 394 have a maximum at least that large, so the global p-value is 0.7884 and the trials factor is 31.58. The median toy maximum is 2.25. The observed maximum is a typical value for a scan with 87 test masses and no signal.

The global number is the one that matters, because the scan looks at 87 places and a fluctuation of about two standard deviations is expected somewhere. The background-only goodness-of-fit p-value is 0.9361, so the smooth four-parameter background describes this region without a visible problem. I do not read anything into the fluctuation at 26.0 GeV.

## 9. What this shows and what it does not

**What this shows.**
- With the nominal selection and model, the fitted Z mass is 90.7905 GeV with a statistical error of 0.0398 GeV and an indicative systematic estimate of 0.0548 GeV. It is 0.3974 GeV below the reference value of 91.1879 GeV, which is 5.86 total standard errors. It does not agree with the reference within my uncertainties.
- The nine variants of the fit change the mass by at most 0.0499 GeV. The difference from the reference is much larger than any of them.
- The J/psi mass comes out 0.111 percent low and the Z mass 0.436 percent low. The offset is not the same relative size at the two masses.
- The scan of 15 to 60 GeV finds a largest local significance of 1.961, with a global p-value of 0.7884 after accounting for the 87 test masses.

**What this does not show.**
- It does not show why the Z mass is low. Final-state radiation, an uncalibrated muon momentum scale, the Drell-Yan line shape and the difference between a fitted peak and the reference definition are all likely contributors, and I did not separate them.
- The systematic estimate covers only my modelling choices. It does not cover detector calibration, efficiency or acceptance, because this file has no simulation.
- The toy study tests the fit code against its own model. It does not test the model against nature.
- The scan result says nothing about particles. The background model is an empirical smooth function and the data are 100000 preselected events from one run period.
- This is an educational analysis of open data and not a CMS result.